# Muon Ablation Demo

This notebook reproduces a minimal end-to-end workflow:

1. Clone the repository
2. Install dependencies with `uv`
3. Run **6 training setups** — 3 architectures × 2 optimizers (AdamW, Muon)
4. Review results and suggested follow-up experiments

> **Tip:** Demo overrides use small data and few epochs. For real runs, see [README.md](../README.md).

In [ ]:
import os
os.chdir("/content/")
!rm -rf muon_ablation
!git clone https://github.com/MichaelNotDeveloper/muon_ablation.git
os.chdir("muon_ablation")

In [ ]:
!pip install uv
!uv sync

## Common demo overrides

Fast smoke-test settings (same idea as `$COMMON` in ablation scripts):

```bash
COMMON="trainer.n_epochs=2 trainer.epoch_len=50 trainer.override=true datasets.val.limit=200 writer.mode=offline batch_size=16 num_workers=2"
```

| # | Config | Optimizer | Run name |
|---|--------|-----------|----------|
| 1 | `lstm` | AdamW | `lstm_adamw` |
| 2 | `lstm` | Muon | `lstm_muon` |
| 3 | `transformer` | AdamW | `transformer_adamw` |
| 4 | `transformer` | Muon | `transformer_muon` |
| 5 | `nanogpt` | AdamW | `nanogpt_adamw` |
| 6 | `nanogpt` | Muon | `nanogpt_muon` |

In [ ]:
!uv run train.py --config-name=lstm optimizer=adamw writer.run_name=lstm_adamw trainer.n_epochs=2 trainer.epoch_len=50 trainer.override=true datasets.val.limit=200 writer.mode=offline batch_size=16 num_workers=2

In [ ]:
!uv run train.py --config-name=lstm optimizer=muon writer.run_name=lstm_muon trainer.n_epochs=2 trainer.epoch_len=50 trainer.override=true datasets.val.limit=200 writer.mode=offline batch_size=16 num_workers=2

In [ ]:
!uv run train.py --config-name=transformer optimizer=adamw writer.run_name=transformer_adamw trainer.n_epochs=2 trainer.epoch_len=50 trainer.override=true datasets.val.limit=200 writer.mode=offline batch_size=16 num_workers=2

In [ ]:
!uv run train.py --config-name=transformer optimizer=muon writer.run_name=transformer_muon trainer.n_epochs=2 trainer.epoch_len=50 trainer.override=true datasets.val.limit=200 writer.mode=offline batch_size=16 num_workers=2

In [ ]:
!uv run train.py --config-name=nanogpt optimizer=adamw writer.run_name=nanogpt_adamw trainer.n_epochs=2 trainer.epoch_len=50 trainer.override=true datasets.val.limit=200 writer.mode=offline batch_size=16 num_workers=2

In [ ]:
!uv run train.py --config-name=nanogpt optimizer=muon writer.run_name=nanogpt_muon trainer.n_epochs=2 trainer.epoch_len=50 trainer.override=true datasets.val.limit=200 writer.mode=offline batch_size=16 num_workers=2

## Inspect saved runs

Checkpoints and Hydra configs are written under `saved/<run_name>/`.

In [ ]:
!ls -la saved/
!find saved -maxdepth 2 -name "*.pth" | head -20

## Suggested follow-up experiments

Uncomment and run any command below (see also [README.md](../README.md)).

In [ ]:
# Learning-rate sweep
# !uv run train.py --config-name=lstm optimizer=adamw optimizer.lr=3e-4 writer.run_name=lstm_adamw_lr3e4 trainer.n_epochs=2 trainer.epoch_len=50 writer.mode=offline
# !uv run train.py --config-name=lstm optimizer=muon optimizer.muon.lr=0.02 writer.run_name=lstm_muon_lr0.02 trainer.n_epochs=2 trainer.epoch_len=50 writer.mode=offline

# Muon projection: exact vs Newton–Schulz
# !uv run train.py --config-name=transformer optimizer=muon optimizer.muon.projection=exact writer.run_name=transformer_muon_exact trainer.n_epochs=2 trainer.epoch_len=50 writer.mode=offline
# !uv run train.py --config-name=transformer optimizer=muon optimizer.muon.projection=ns writer.run_name=transformer_muon_ns trainer.n_epochs=2 trainer.epoch_len=50 writer.mode=offline

# Momentum ablation
# !uv run train.py --config-name=lstm optimizer=muon optimizer.muon.momentum=0.9 optimizer.muon.nesterov=true writer.run_name=lstm_muon_mom0.9 trainer.n_epochs=2 trainer.epoch_len=50 writer.mode=offline

# Scale up
# !uv run train.py --config-name=nanogpt optimizer=muon datasets.train.download_limit=50000 trainer.n_epochs=20 trainer.epoch_len=500 writer.run_name=nanogpt_muon_full writer.mode=offline

### Ideas to explore next

1. **Learning-rate sweep** — best LR per (architecture, optimizer)
2. **Muon projection** — `optimizer.muon.projection=exact` vs `ns`
3. **Momentum / Nesterov** — `optimizer.muon.momentum=0.9 optimizer.muon.nesterov=true`
4. **Matrix metrics** — compare `condition_number_weighted_mean`, `orthogonality_error_weighted_mean`, `spectral_norm_weighted_mean` in logs
5. **Scale up** — increase `datasets.train.download_limit`, `trainer.n_epochs`, model size in `src/configs/model/`
6. **Online logging** — `writer.mode=online` + W&B project settings